# CTGAN notebook demo

We split the sample transactions into 60% training, 20% validation, and 20% test subsets,
train SDV's CTGAN on the training split, and compare the generated output
against every subset plus the pipeline's default diagnostics.


In [ ]:
import csv
import json
import sys
from pathlib import Path

from IPython.display import Image, JSON, display

repo_root = Path.cwd().resolve().parents[0]
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from transaction_gan.config import SchemaConfig
from transaction_gan.gan import GANTrainingConfig
from transaction_gan.pipeline import generate_synthetic_transactions
from transaction_gan.data_loader import load_transactions
from transaction_gan.data_split import split_records
from transaction_gan.evaluation import compare_statistics
from transaction_gan.testing import generate_testing_report


In [ ]:
data_path = repo_root / "data" / "sample_transactions.csv"
output_path = repo_root / "data" / "synthetic_transactions_notebook.csv"
split_seed = 123

records = load_transactions(data_path)
train_records, remaining_records = split_records(records, train_fraction=0.6, seed=split_seed)
validation_records, test_records = split_records(remaining_records, train_fraction=0.5, seed=split_seed + 1)

print(f"Train rows: {len(train_records)}")
print(f"Validation rows: {len(validation_records)}")
print(f"Test rows: {len(test_records)}")


In [ ]:
schema = SchemaConfig(
    id_column="transaction_id",
    continuous_columns=["amount", "customer_age"],
    categorical_columns=[
        "merchant_category",
        "transaction_type",
        "merchant_code",
        "is_fraud",
        "transaction_date",
        "transaction_address",
    ],
    hierarchical_categorical_groups=[("merchant_category", "transaction_type")],
    drop_columns=["transaction_id"],
)

schema


In [ ]:
gan_config = GANTrainingConfig(
    noise_dim=128,
    hidden_dim=256,
    epochs=300,
    learning_rate=2e-4,
    batch_size=512,
)

gan_config


In [ ]:
result = generate_synthetic_transactions(
    data_path=data_path,
    output_path=output_path,
    schema=schema,
    gan_config=gan_config,
    samples_to_generate=256,
    train_fraction=0.6,
    split_seed=split_seed,
)

display(JSON(result["quality_report"]))
display(result["synthetic_preview"])

print("Synthetic CSV stored at:", result["synthetic_path"])
print("Metrics JSON stored at:", result["metrics_path"])
print("Testing report stored at:", result["testing_report_path"])

display(Image(filename=result["visualization_path"]))
display(Image(filename=result["training_history_path"]))
display(Image(filename=result["testing_visualization_path"]))


In [ ]:
result["split_summary"]


In [ ]:
history = result["training_history"]
history[:5]


In [ ]:
with open(result["synthetic_path"], "r", encoding="utf8") as handle:
    synthetic_records = list(csv.DictReader(handle))

len(synthetic_records)


In [ ]:
train_eval = compare_statistics(train_records, synthetic_records, numeric_features=schema.continuous_columns)
validation_eval = compare_statistics(validation_records, synthetic_records, numeric_features=schema.continuous_columns)
test_eval = compare_statistics(test_records, synthetic_records, numeric_features=schema.continuous_columns)

display(JSON({"train": train_eval.to_dict(), "validation": validation_eval.to_dict(), "test": test_eval.to_dict()}))


In [ ]:
manual_testing = {
    "train": generate_testing_report(train_records, synthetic_records, schema=schema),
    "validation": generate_testing_report(validation_records, synthetic_records, schema=schema),
    "test": generate_testing_report(test_records, synthetic_records, schema=schema),
}

JSON(manual_testing)


In [ ]:
testing_report = json.loads(Path(result["testing_report_path"]).read_text())
testing_report
